# EmbracePlus Raw Data Batch Downloader — user-wise, all modalities

This notebook:

1. Connects to your Empatica S3 bucket using your Access Key.
2. Scans every account (device) from **March 2026 onward**.
3. Matches each day's data to the right **anonymized user folder** (user1, user2, ... — no real names anywhere) using the session table below.
4. Extracts **all 10 raw modalities** (accelerometer, gyroscope, magnetometer, eda, temperature, bvp, systolicPeaks, steps, tags, ambientLight) with correct unit conversions.
5. Saves one CSV per modality, per session, into `output/userN/`, each tagged with account number, date, and both UTC + IST timestamps.

**Before running:** fill in Cell 2 (credentials) and check Cell 4 (org/study/site IDs) and Cell 5 (session table) match your setup. Rows marked `# REVIEW` came from ambiguous entries in your original table — double check them.


In [1]:
# Cell 1 — install dependencies (run once)
%pip install boto3 fastavro pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 2 — YOUR CREDENTIALS (fill these in)
ACCESS_KEY = os.environ["AWS_ACCESS_KEY_ID"]
SECRET_KEY = os.environ["AWS_SECRET_ACCESS_KEY"]
REGION     = "us-east-1"

BUCKET = "empatica-us-east-1-prod-data"


In [2]:
# Cell 3 — S3 client setup
import boto3

s3 = boto3.client(
    "s3",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION,
)
print("S3 client ready.")


S3 client ready.


In [3]:
# Cell 4 — org / study / site path (from your S3 Access URL, e.g. s3://.../v2/2753/1/1/...)
ORG_ID   = "2753"
STUDY_ID = "1"
SITE_ID  = "1"

BASE_PREFIX = f"v2/{ORG_ID}/{STUDY_ID}/{SITE_ID}"
print("Base prefix:", BASE_PREFIX)


Base prefix: v2/2753/1/1


In [ ]:
# Cell 5 — SESSION TABLE, cut down to the four users we actually need
#
# BTP CHANGE. The original table lists all 21 users. Downloading them all takes
# a long time and a lot of disk. We were decided to use only:
#
#     user16, user17, user3, user20
#
# Everything below is copied from the original rows for those four, unchanged.
# The other rows are not deleted from the project, they are just not scanned:
# the untouched original is kept in mentor_files/.
#
# Dates are 'YYYY-MM-DD'. Times are 'HH:MM' in 24-hour IST.
# None means "not specified / full day / open end".

SESSION_TABLE = [
    # user,    account, start_date,   end_date,      start_time, end_time, exclude
    ("user3",  0, "2026-05-18", None,           "16:08", "00:00", False),
    ("user3",  0, "2026-07-19", None,           "16:40", "00:00", False),
    ("user3",  0, "2026-07-20", None,           "13:00", "00:00", False),
    ("user3",  0, "2026-07-21", None,           "12:00", "00:12", False),
    ("user3",  0, "2026-07-22", None,           "10:00", "15:09", False),
    ("user3",  0, "2026-07-22", None,           "16:00", "23:59", False),  # second session same day
    ("user3",  0, "2026-07-26", None,           "14:00", "22:00", False),
    ("user16", 6, "2026-06-23", "2026-06-30",   None,    None,    False),  # whole range, account-level
    ("user17", 6, "2026-07-01", "2026-07-09",   None,    None,    False),  # whole range, account-level
    ("user20", 6, "2026-08-12", "2026-08-13",   None,    None,    False),
]

# BTP CHANGE. The fallback is switched off.
#
# Originally this was {0: "user2", 2: "user18"}, which means any day on
# account 0 that nobody claimed still got saved as user2, and the whole of
# account 2 got saved as user18. That is most of the download volume, and none
# of it is one of our four users. With the fallback empty, unclaimed data is
# simply skipped.
FALLBACK_BY_ACCOUNT = {}

# The four users we keep. Used again in the main loop as a final guard.
KEEP_USERS = {"user3", "user16", "user17", "user20"}

print("Session rows kept:", len(SESSION_TABLE))
print("Users:", sorted(KEEP_USERS))


Session rows kept: 10
Users: ['user16', 'user17', 'user20', 'user3']


In [5]:
# Cell 6 — date helpers
import pandas as pd
from datetime import datetime, timedelta

IST_OFFSET = pd.Timedelta(hours=5, minutes=30)
SCAN_START_DATE = pd.Timestamp("2026-03-01")   # scan from March 2026 onward
SCAN_END_DATE   = pd.Timestamp.today().normalize()

def daterange(start, end):
    days = []
    d = start
    while d <= end:
        days.append(d.strftime("%Y-%m-%d"))
        d += timedelta(days=1)
    return days

def parse_hhmm(date_str, hhmm):
    """Combine a date string and 'HH:MM' (IST) into a UTC Timestamp."""
    if hhmm is None:
        return None
    ist_dt = pd.Timestamp(f"{date_str} {hhmm}:00")
    return ist_dt - IST_OFFSET


In [6]:
# Cell 7 — figure out which account+date windows belong to which user
# Builds a lookup: for a given (account, date), a list of (user, start_utc_or_None, end_utc_or_None, whole_day)

def build_ownership_index():
    idx = {}
    for user, account, sdate, edate, stime, etime, exclude in SESSION_TABLE:
        if exclude:
            continue
        if account == 2:
            # account 2 belongs entirely to this user, no date filtering
            idx.setdefault(("ALL_DATES", account), []).append((user, None, None, True))
            continue
        end_date = edate if edate else sdate
        for date_str in daterange(pd.Timestamp(sdate), pd.Timestamp(end_date)):
            whole_day = (stime is None and etime is None)
            start_utc = parse_hhmm(date_str, stime) if stime else None
            # handle overnight end times (e.g. '00:00' or times earlier than start) by rolling to next day
            end_utc = None
            if etime:
                end_utc = parse_hhmm(date_str, etime)
                if start_utc is not None and end_utc <= start_utc:
                    end_utc += pd.Timedelta(days=1)
            idx.setdefault((date_str, account), []).append((user, start_utc, end_utc, whole_day))
    return idx

OWNERSHIP_INDEX = build_ownership_index()
print(f"Built ownership index for {len(OWNERSHIP_INDEX)} (date, account) keys.")


Built ownership index for 25 (date, account) keys.


In [7]:
# Cell 8 — S3 listing helpers

def list_participant_folders(date_str):
    """List participant subfolders (e.g. '0006-3YK9J1H1C7') under participant_data/{date}/"""
    prefix = f"{BASE_PREFIX}/participant_data/{date_str}/"
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, Delimiter="/")
    folders = []
    for cp in resp.get("CommonPrefixes", []):
        name = cp["Prefix"].rstrip("/").split("/")[-1]   # e.g. '0006-3YK9J1H1C7'
        folders.append(name)
    return folders

def account_from_folder(folder_name):
    """'0006-3YK9J1H1C7' -> account number 6"""
    try:
        return int(folder_name.split("-")[0])
    except Exception:
        return None

def list_avro_files(date_str, participant_folder):
    prefix = f"{BASE_PREFIX}/participant_data/{date_str}/{participant_folder}/raw_data/v6/"
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    return [obj["Key"] for obj in resp.get("Contents", [])]


In [8]:
# Cell 9 — owner lookup for a given account/date/timestamp

def find_owner(account, date_str, sample_utc_ts):
    # account 2 -> always user18
    if ("ALL_DATES", account) in OWNERSHIP_INDEX:
        return OWNERSHIP_INDEX[("ALL_DATES", account)][0][0]

    candidates = OWNERSHIP_INDEX.get((date_str, account), [])
    for user, start_utc, end_utc, whole_day in candidates:
        if whole_day:
            return user
        if start_utc is not None and end_utc is not None:
            if start_utc <= sample_utc_ts <= end_utc:
                return user
        elif start_utc is not None and end_utc is None:
            if sample_utc_ts >= start_utc:
                return user

    # fallback rules
    fallback_user = FALLBACK_BY_ACCOUNT.get(account)
    if fallback_user:
        return fallback_user

    return f"unassigned_account{account}"


In [9]:
# Cell 10 — extract ALL 10 raw modalities from one avro file, with unit conversion + UTC/IST timestamps

from fastavro import reader as avro_reader
import io

FLAT_SENSORS = ["eda", "temperature", "bvp", "steps", "ambientLight"]
AXIS_SENSORS = ["accelerometer", "gyroscope", "magnetometer"]
TIMESTAMP_ONLY = ["systolicPeaks", "tags"]

def extract_all_modalities(file_bytes, account, date_str):
    """Returns dict: sensor_name -> DataFrame (with account_number, date, datetime_utc, datetime_ist columns)"""
    result = {}
    buf = io.BytesIO(file_bytes)
    for record in avro_reader(buf):
        raw = record["rawData"]

        # --- flat single-value sensors ---
        for sensor in FLAT_SENSORS:
            s = raw.get(sensor)
            if not s or s.get("samplingFrequency", 0) == 0:
                continue
            fs = s["samplingFrequency"]
            start_ts = s["timestampStart"]
            n = len(s["values"])
            ts_utc = pd.to_datetime([start_ts + i * (1e6 / fs) for i in range(n)], unit="us")
            df = pd.DataFrame({"datetime_utc": ts_utc, sensor: s["values"]})
            df["datetime_ist"] = df["datetime_utc"] + IST_OFFSET
            df["account_number"] = account
            df["date"] = date_str
            result[sensor] = df

        # --- 3-axis sensors ---
        for sensor in AXIS_SENSORS:
            s = raw.get(sensor)
            if not s or s.get("samplingFrequency", 0) == 0:
                continue
            fs = s["samplingFrequency"]
            start_ts = s["timestampStart"]
            n = len(s["x"])
            ts_utc = pd.to_datetime([start_ts + i * (1e6 / fs) for i in range(n)], unit="us")

            if sensor in ("accelerometer", "gyroscope"):
                params = s["imuParams"]
                cf = params.get("conversionFactor")
                if cf:
                    x = [v * cf for v in s["x"]]
                    y = [v * cf for v in s["y"]]
                    z = [v * cf for v in s["z"]]
                else:
                    dphys = params["physicalMax"] - params["physicalMin"]
                    ddig = params["digitalMax"] - params["digitalMin"]
                    x = [v * dphys / ddig for v in s["x"]]
                    y = [v * dphys / ddig for v in s["y"]]
                    z = [v * dphys / ddig for v in s["z"]]
                df = pd.DataFrame({
                    "datetime_utc": ts_utc,
                    f"{sensor}_x": x, f"{sensor}_y": y, f"{sensor}_z": z,
                })
            else:  # magnetometer — already physical units, has quality index
                df = pd.DataFrame({
                    "datetime_utc": ts_utc,
                    "magnetometer_x": s["x"], "magnetometer_y": s["y"], "magnetometer_z": s["z"],
                    "quality_index": s.get("qualityIndex", [None]*n),
                })

            df["datetime_ist"] = df["datetime_utc"] + IST_OFFSET
            df["account_number"] = account
            df["date"] = date_str
            result[sensor] = df

        # --- timestamp-only sensors ---
        sp = raw.get("systolicPeaks")
        if sp and sp.get("peaksTimeNanos"):
            ts_utc = pd.to_datetime(sp["peaksTimeNanos"], unit="ns")
            df = pd.DataFrame({"datetime_utc": ts_utc})
            df["datetime_ist"] = df["datetime_utc"] + IST_OFFSET
            df["account_number"] = account
            df["date"] = date_str
            result["systolicPeaks"] = df

        tags = raw.get("tags")
        if tags and tags.get("tagsTimeMicros"):
            ts_utc = pd.to_datetime(tags["tagsTimeMicros"], unit="us")
            df = pd.DataFrame({"datetime_utc": ts_utc})
            df["datetime_ist"] = df["datetime_utc"] + IST_OFFSET
            df["account_number"] = account
            df["date"] = date_str
            result["tags"] = df

        break  # one record per avro file
    return result


In [10]:
# Cell 11 — split extracted sensor data by owner (a file/date can belong to multiple users if times split within a day)

def split_by_owner(sensor_dfs, account, date_str):
    """Returns dict: user -> {sensor_name: DataFrame subset for that user}"""
    out = {}
    for sensor, df in sensor_dfs.items():
        if df.empty:
            continue
        owners = df["datetime_utc"].apply(lambda ts: find_owner(account, date_str, ts))
        for user in owners.unique():
            sub = df[owners == user]
            out.setdefault(user, {})[sensor] = sub
    return out


In [17]:
# Cell 12 — save helper: append per-user, per-sensor CSVs to output/userN/

import os

# The notebook runs from btp/, so a plain relative path lands in
# btp/data/... instead of the project's own data folder. Going up one level
# fixes it. The print confirms where it actually resolved to, so this cannot
# go wrong silently again.
OUTPUT_DIR = os.path.abspath(os.path.join("..", "data", "raw", "embraceplus"))
print("Saving to:", OUTPUT_DIR)

def save_user_sensor_data(user, account, date_str, sensor, df):
    user_dir = os.path.join(OUTPUT_DIR, user)
    os.makedirs(user_dir, exist_ok=True)
    fname = f"{user}_acct{account}_{date_str}_{sensor}.csv"
    fpath = os.path.join(user_dir, fname)
    write_header = not os.path.exists(fpath)
    df.to_csv(fpath, mode="a", index=False, header=write_header)


Saving to: /mnt/e/CSE_Skills/Sem_7/BTP/Physiological-Stress-Detection/AdaStress_Integration/AdaStress_BTP/data/raw/embraceplus


In [18]:
# Cell 12b — resume / skip-already-processed manifest

MANIFEST_PATH = os.path.join(OUTPUT_DIR, "processed_files.log")

def load_manifest():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if not os.path.exists(MANIFEST_PATH):
        return set()
    with open(MANIFEST_PATH, "r") as f:
        return set(line.strip() for line in f if line.strip())

def mark_processed(key):
    # append immediately so a crash/network drop never loses progress
    with open(MANIFEST_PATH, "a") as f:
        f.write(key + "\n")

PROCESSED_KEYS = load_manifest()
print(f"{len(PROCESSED_KEYS)} files already processed in previous runs — these will be skipped.")


0 files already processed in previous runs — these will be skipped.


In [19]:
# Cell 13 — MAIN LOOP (resumable): scan every date from March 2026 onward, skip already-processed files, download, extract, assign, save

import time
from botocore.exceptions import EndpointConnectionError, ClientError

MAX_RETRIES = 5
RETRY_WAIT_SECONDS = 5

# BTP CHANGE. Scan only the dates the four users actually wore the device,
# instead of every date from March 2026 to today.
#
# The original line was:
#     all_dates = daterange(SCAN_START_DATE, SCAN_END_DATE)
# which is around 200 dates. The four users cover about 20. Everything else
# would be downloaded, unpacked and then thrown away.

wanted_dates = set()
for _user, _acct, _sd, _ed, _st, _et, _excl in SESSION_TABLE:
    if _excl or _sd is None:
        continue
    wanted_dates.update(daterange(pd.Timestamp(_sd), pd.Timestamp(_ed or _sd)))

all_dates = sorted(wanted_dates)
# all_dates = all_dates[:1]   # TEMPORARY, remove after the test
print(f"Scanning {len(all_dates)} dates for {len(KEEP_USERS)} users "
      f"(the original scanned every date from March 2026).")
print("Dates:", all_dates)
processed_files = 0
skipped_already_done = 0
skipped_dates = 0

for date_str in all_dates:
    folders = list_participant_folders(date_str)
    if not folders:
        skipped_dates += 1
        continue

    for folder in folders:
        account = account_from_folder(folder)
        if account is None:
            continue

        avro_keys = list_avro_files(date_str, folder)
        if not avro_keys:
            continue

        for key in avro_keys:
            if key in PROCESSED_KEYS:
                skipped_already_done += 1
                continue

            # retry loop for transient network errors
            file_bytes = None
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    obj = s3.get_object(Bucket=BUCKET, Key=key)
                    file_bytes = obj["Body"].read()
                    break
                except (EndpointConnectionError, ClientError) as e:
                    print(f"  [retry {attempt}/{MAX_RETRIES}] network issue on {key}: {e}")
                    time.sleep(RETRY_WAIT_SECONDS)
            else:
                print(f"  [SKIPPED after {MAX_RETRIES} retries] {key}")
                continue  # give up on this file, move to next, but do NOT mark as processed

            sensor_dfs = extract_all_modalities(file_bytes, account, date_str)
            if not sensor_dfs:
                mark_processed(key)
                PROCESSED_KEYS.add(key)
                continue

            owner_split = split_by_owner(sensor_dfs, account, date_str)

            for user, sensors in owner_split.items():
                # BTP CHANGE. Belt and braces: even if a row slips through the
                # ownership index, nothing outside the four users is written.
                if user not in KEEP_USERS:
                    continue
                for sensor, df in sensors.items():
                    save_user_sensor_data(user, account, date_str, sensor, df)

            mark_processed(key)
            PROCESSED_KEYS.add(key)
            processed_files += 1

            if processed_files % 20 == 0:
                print(f"Processed {processed_files} new files so far... (currently on {date_str})")

print(f"\nDone. Processed {processed_files} new avro files. "
      f"Skipped {skipped_already_done} already-done files. "
      f"{skipped_dates} dates had no data at all.")
print("You can safely re-run this cell any time — it will pick up where it left off.")


Scanning 25 dates for 4 users (the original scanned every date from March 2026).
Dates: ['2026-05-18', '2026-06-23', '2026-06-24', '2026-06-25', '2026-06-26', '2026-06-27', '2026-06-28', '2026-06-29', '2026-06-30', '2026-07-01', '2026-07-02', '2026-07-03', '2026-07-04', '2026-07-05', '2026-07-06', '2026-07-07', '2026-07-08', '2026-07-09', '2026-07-19', '2026-07-20', '2026-07-21', '2026-07-22', '2026-07-26', '2026-08-12', '2026-08-13']
Processed 20 new files so far... (currently on 2026-05-18)
Processed 40 new files so far... (currently on 2026-05-18)
Processed 60 new files so far... (currently on 2026-06-23)
Processed 80 new files so far... (currently on 2026-06-23)
Processed 100 new files so far... (currently on 2026-06-24)
Processed 120 new files so far... (currently on 2026-06-24)
Processed 140 new files so far... (currently on 2026-06-24)
Processed 160 new files so far... (currently on 2026-06-25)
Processed 180 new files so far... (currently on 2026-06-25)
Processed 200 new files s

In [16]:
# Cell 13c — DIAGNOSTIC. Why did a date produce no user files?
# Reads a single date, ignores the manifest, saves nothing.

DEBUG_DATE = "2026-05-18"

folders = list_participant_folders(DEBUG_DATE)
print(f"{DEBUG_DATE}: {len(folders)} participant folders")
for f in folders:
    print(f"   {f}  -> account {account_from_folder(f)}")

print()
print("Session rows we have for this date:")
for acct in sorted({account_from_folder(f) for f in folders if account_from_folder(f) is not None}):
    rows = OWNERSHIP_INDEX.get((DEBUG_DATE, acct), [])
    if not rows:
        print(f"   account {acct}: NO session row -> everything unassigned")
    for user, s, e, whole in rows:
        print(f"   account {acct}: {user}  window {s} .. {e} UTC  (whole_day={whole})")

print()
for f in folders:
    acct = account_from_folder(f)
    keys = list_avro_files(DEBUG_DATE, f)
    if not keys:
        continue
    obj = s3.get_object(Bucket=BUCKET, Key=keys[0])
    dfs = extract_all_modalities(obj["Body"].read(), acct, DEBUG_DATE)
    if not dfs:
        print(f"   {f}: first file has no usable modalities")
        continue
    sensor = list(dfs.keys())[0]
    df = dfs[sensor]
    print(f"   {f} (account {acct}), {len(keys)} avro files")
    print(f"      first file '{sensor}': {df['datetime_utc'].min()} .. "
          f"{df['datetime_utc'].max()} UTC")
    print(f"                              {df['datetime_ist'].min()} .. "
          f"{df['datetime_ist'].max()} IST")
    owners = df["datetime_utc"].apply(lambda ts: find_owner(acct, DEBUG_DATE, ts))
    print(f"      owners assigned: {dict(owners.value_counts())}")

2026-05-18: 2 participant folders
   0000-3YK9J1H1C7  -> account 0
   0004-3YK9J1H1C7  -> account 4

Session rows we have for this date:
   account 0: user3  window 2026-05-18 10:38:00 .. 2026-05-18 18:30:00 UTC  (whole_day=False)
   account 4: NO session row -> everything unassigned

   0000-3YK9J1H1C7 (account 0), 40 avro files
      first file 'eda': 2026-05-18 00:29:29.388981 .. 2026-05-18 00:59:34.138765750 UTC
                              2026-05-18 05:59:29.388981 .. 2026-05-18 06:29:34.138765750 IST
      owners assigned: {'unassigned_account0': 7220}
   0004-3YK9J1H1C7 (account 4), 6 avro files
      first file 'eda': 2026-05-18 20:09:02.396314 .. 2026-05-18 20:39:08.146314 UTC
                              2026-05-19 01:39:02.396314 .. 2026-05-19 02:09:08.146314 IST
      owners assigned: {'unassigned_account4': 7224}


## Notes / things to double-check after running

- Rows marked `# REVIEW` in Cell 5 came from ambiguous entries in your original table (crossing-midnight times written inconsistently, a possible duplicate account for `user14`, an unclear second session for `user3` on 22/07). Verify these against your original records.
- Any account/date combination not covered by an explicit row and not account 0 or 2 will be saved under `unassigned_accountN` — check the `output/` folder for any such folders and assign them manually if needed.
- All output filenames follow the pattern `userN_acctK_YYYY-MM-DD_sensorname.csv` — no real names anywhere, only anonymized user IDs, account numbers, and dates.
- Every row inside every CSV carries `account_number`, `date`, `datetime_utc`, and `datetime_ist` columns for full traceability.
